In [ ]:
# %pip install faster-whisper speechbrain opensmile sounddevice soundfile numpy scipy torch torchaudio transformers librosa

In [ ]:
import queue
import tempfile
import threading
import time
import os

import numpy as np
import sounddevice as sd
import soundfile as sf

from faster_whisper import WhisperModel

import opensmile

from transformers import pipeline

In [ ]:
SAMPLE_RATE = 16000
CHANNELS = 1

# tempo de cada chunk de áudio
CHUNK_DURATION = 10  # segundos

# pasta temporária
TEMP_DIR = "temp_audio"

os.makedirs(TEMP_DIR, exist_ok=True)

In [ ]:
audio_queue = queue.Queue()

In [ ]:
print("Carregando Whisper...")

whisper_model = WhisperModel(
    "small",
    device="cpu",      # troque para "cpu" se necessário
    compute_type="int8"
)

print("Whisper carregado.")

In [ ]:
print("Carregando modelo de emoção...")

emotion_classifier = pipeline(
    "audio-classification",
    model="ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition"
)

print("SpeechBrain carregado.")

In [ ]:

print("Carregando openSMILE...")

smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

print("openSMILE carregado.")

In [ ]:
recorded_chunks = []

last_save_time = time.time()


def audio_callback(indata, frames, time_info, status):

    global recorded_chunks
    global last_save_time

    if status:
        print(status)

    recorded_chunks.append(indata.copy())

    elapsed = time.time() - last_save_time

    if elapsed >= CHUNK_DURATION:

        audio_data = np.concatenate(recorded_chunks, axis=0)

        audio_queue.put(audio_data)

        recorded_chunks = []

        last_save_time = time.time()

# ============================================================
# PROCESSAMENTO
# ============================================================

In [ ]:
def process_audio():

    while True:

        audio_data = audio_queue.get()

        try:

            # ==========================================
            # NORMALIZA ÁUDIO
            # ==========================================

            audio_np = audio_data.flatten()

            max_val = np.max(np.abs(audio_np))

            if max_val > 0:
                audio_np = audio_np / max_val

            # ==========================================
            # SALVA TEMP
            # ==========================================

            temp_file = tempfile.NamedTemporaryFile(
                suffix=".wav",
                delete=False,
                dir=TEMP_DIR
            )

            sf.write(
                temp_file.name,
                audio_np,
                SAMPLE_RATE
            )

            wav_path = temp_file.name

            print("\n===================================================")
            print("NOVO CHUNK PROCESSADO")
            print("Arquivo:", wav_path)

            # ==========================================
            # WHISPER
            # ==========================================

            print("\n[TRANSCRIÇÃO]")

            segments, info = whisper_model.transcribe(
                audio_np,
                language="pt"
            )

            full_text = ""

            for segment in segments:
                full_text += segment.text + " "

            full_text = full_text.strip()

            print("Texto:", full_text)

            # ==========================================
            # EMOTION
            # ==========================================

            print("\n[EMOÇÃO]")

            result = emotion_classifier(
                audio_np,
                sampling_rate=16000
            )

            for item in result:
                print(
                    f"{item['label']} -> {item['score']:.4f}"
                )

            top_emotion = max(
                result,
                key=lambda x: x["score"]
            )

            print(
                f"\nEmoção principal: {top_emotion['label']}"
            )

            # ==========================================
            # ANÁLISE EMOCIONAL
            # ==========================================

            emotion_label = top_emotion["label"].lower()
            emotion_score = top_emotion["score"]

            if emotion_label == "sad" and emotion_score > 0.5:
                print("Indício forte de tristeza.")

            elif emotion_label == "fearful":
                print("Possível ansiedade ou tensão.")

            elif emotion_label == "angry":
                print("Possível irritabilidade.")

            elif emotion_label == "happy":
                print("Tom emocional positivo.")

            elif emotion_label == "neutral":
                print("Tom emocional neutro.")

            # ==========================================
            # OPENSMILE FEATURES
            # ==========================================

            print("\n[FEATURES VOCAIS]")

            features = smile.process_file(wav_path)

            selected_features = [
                "F0semitoneFrom27.5Hz_sma3nz_amean",
                "loudness_sma3_amean",
                "alphaRatioV_sma3nz_amean",
                "hammarbergIndexV_sma3nz_amean",
                "mfcc1_sma3_amean",
            ]

            for feature in selected_features:

                if feature in features.columns:

                    value = features[feature].values[0]

                    print(f"{feature}: {value}")

            # ==========================================
            # ANÁLISE SIMPLEX
            # ==========================================

            print("\n[ANÁLISE SIMPLEX]")

            loudness = None

            if "loudness_sma3_amean" in features.columns:

                loudness = features[
                    "loudness_sma3_amean"
                ].values[0]

            if loudness is not None:

                if loudness < 0.2:
                    print(
                        "Possível baixa energia vocal."
                    )

            # ==========================================
            # ANÁLISE COMBINADA
            # ==========================================

            if (
                emotion_label == "sad"
                and loudness is not None
                and loudness < 0.2
            ):
                print(
                    "Possível combinação de tristeza + baixa energia vocal."
                )

            # ==========================================
            # LIMPA TEMP
            # ==========================================

            os.remove(wav_path)

        except Exception as e:
            print("Erro:", e)

# ============================================================
# THREAD PROCESSAMENTO
# ============================================================

In [ ]:
processing_thread = threading.Thread(
    target=process_audio,
    daemon=True
)

processing_thread.start()

# ============================================================
# START MICROFONE
# ============================================================

In [ ]:
print("\n===================================================")
print("🎤 Ouvindo microfone em tempo real...")
print("Pressione CTRL+C para parar.")
print("===================================================\n")

try:

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=CHANNELS,
        callback=audio_callback,
    ):

        while True:
            time.sleep(0.1)

except KeyboardInterrupt:

    print("\nEncerrado.")